# Post-processing of recorded spectra

Load the on/off spectrum pairs recorded by `rtlsdr-recorder`, clean and
downsample them, average them, and export the result to a FITS spectrum.

The recorder saves frequency-switched pairs of one-second integrations: an
*on* spectrum centered on the HI line at 1420 MHz and an *off* spectrum
centered 4 MHz below, so that the two bands do not overlap and the off band
is free of Galactic HI emission.

The difference spectrum can be reduced in two ways, which are described in the two sections below respectively.
In each section, there are three sub-sections which show:
* An outline of the algorithm
* The step-by-step reduction of the data following the algorithm
* An example of how the same steps can be run with a single convenience ``reduce_spectra`` function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rtlsdr_recorder import frequency_array
from rtlsdr_recorder.analysis import (
    load_spectrum_pairs,
    clean_spectrum,
    downsample_spectrum,
    average_spectra,
    plot_spectrum_pair,
    reduce_spectra,
    write_fits,
)

In [ ]:
OUTPUT_DIRECTORY = '../lna-cable-rtl-usb-wifi-off/'

## Reduction with `clip_difference=False` (the default)

### Algorithm

1. Load all matched on/off spectrum pairs from the data directory.
2. Mask RFI spikes in each on and each off spectrum by iterative sigma
   clipping.
3. Mask the channels at the center of each band, which always contain a
   spurious DC spike from the receiver.
4. Spectrally downsample each spectrum by block-averaging groups of
   adjacent channels, ignoring masked values.
5. Average all the on spectra and all the off spectra,
   again ignoring masked values (channels masked in every spectrum, such as
   the DC region, become NaN).
6. Subtract the averaged off spectrum from the averaged on spectrum: since
   both were measured through the same receiver channels, this removes the
   bandpass shape and leaves the HI signal.

### Step by step

Load all matched on/off spectrum pairs:

In [ ]:
spectra_on, spectra_off = load_spectrum_pairs(OUTPUT_DIRECTORY)
len(spectra_on)

Plot the first pair and its difference:

In [ ]:
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

The spectra have RFI spikes, and there is always a spike at the center of
the band, so mask both with `clean_spectrum` (sigma clipping plus a mask on
the central channels):

In [ ]:
spectra_on = [clean_spectrum(spec) for spec in spectra_on]
spectra_off = [clean_spectrum(spec) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

Spectrally downsample the cleaned spectra:

In [ ]:
DOWNSAMPLE = 10
freq = downsample_spectrum(frequency_array(), DOWNSAMPLE)
spectra_on = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_on]
spectra_off = [downsample_spectrum(spec, DOWNSAMPLE) for spec in spectra_off]
plot_spectrum_pair(spectra_on[0], spectra_off[0], frequencies=freq);

That was the first pair only, so now average all pairs:

In [ ]:
average_on = average_spectra(spectra_on)
average_off = average_spectra(spectra_off)
plot_spectrum_pair(average_on, average_off, frequencies=freq);

### The same reduction with `reduce_spectra`

The single call gives the same frequencies, averages, and difference
spectrum as the steps above:

In [ ]:
reduced = reduce_spectra(OUTPUT_DIRECTORY)
np.testing.assert_allclose(reduced.frequencies, freq)
np.testing.assert_allclose(reduced.spectrum_on, average_on)
np.testing.assert_allclose(reduced.spectrum_off, average_off)
np.testing.assert_allclose(reduced.spectrum_diff, average_on - average_off)

## Reduction with `clip_difference=True`

### Algorithm

1. Load all matched on/off spectrum pairs as above.
2. For each pair, subtract the off spectrum from the on spectrum, removing
   the bandpass shape.
3. Mask RFI spikes in each difference spectrum by iterative sigma clipping.
4. Mask the channels at the center of the band as above.
5. Spectrally downsample each difference spectrum.
6. Average all the difference spectra.

The averaged on and off spectra returned by `reduce_spectra` are always
computed as in the previous section; the flag only changes how the
difference spectrum is derived.

### Step by step

Reload the raw pairs (the previous section modified them in place), form
the per-pair differences, then clean, downsample, and average those:

In [ ]:
spectra_on, spectra_off = load_spectrum_pairs(OUTPUT_DIRECTORY)
diffs = [clean_spectrum(on - off) for on, off in zip(spectra_on, spectra_off)]
diffs = [downsample_spectrum(diff, DOWNSAMPLE) for diff in diffs]
average_diff_clipped = average_spectra(diffs)

plt.figure(figsize=(6,2))
plt.plot(freq, average_diff_clipped)
plt.xlabel('Frequency (MHz)');

### The same reduction with `reduce_spectra`

The single call gives the same difference spectrum:

In [ ]:
reduced_clipped = reduce_spectra(OUTPUT_DIRECTORY, clip_difference=True)
np.testing.assert_allclose(reduced_clipped.spectrum_diff, average_diff_clipped)

## Export

Export the averaged difference spectrum from the default reduction to a
FITS file:

In [ ]:
write_fits(average_on - average_off, 'spectrum.fits', frequencies=freq, overwrite=True)